## Setup

In [ ]:
import os

# Define the base directory within Google Drive for your project data
BASE_DIR = '../data'
DATA_SUBFOLDER_NAME = 'N1_tupi_none_short_r1_778989'
FULL_DATA_PATH = os.path.join(BASE_DIR, DATA_SUBFOLDER_NAME)

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(FULL_DATA_PATH, exist_ok=True)

print(f"Permanent 'data' folder created at: {BASE_DIR}")
print(f"Permanent subfolder '{DATA_SUBFOLDER_NAME}' created at: {FULL_DATA_PATH}")

for folder in os.listdir(BASE_DIR):
    print(folder)

Permanent 'data' folder created at: ../data
Permanent subfolder 'N1_tupi_none_short_r1_778989' created at: ../data/N1_tupi_none_short_r1_778989
N2_poti_PP_short_r1_780628
N2_poti_TP_short_r1_780627
N1_tupi_none_long_r1_780636
N4_poti_PP_long_r1_780638
N2_poti_PP_long_r1_780630
N4_poti_PP_short_r1_780634
N4_poti_TP_short_r1_780631
N1_tupi_none_short_r1_778989
N2_poti_TP_long_r1_780629
N1_tupi_none_short_r1_780637
N4_poti_TP_long_r1_780632


In [5]:
FULL_DATA_PATH_MULTI = BASE_DIR

print(f"Caminho atual: {FULL_DATA_PATH_MULTI}")

for folder in os.listdir(FULL_DATA_PATH_MULTI):
    print(folder)

Caminho atual: ../data
N2_poti_PP_short_r1_780628
N2_poti_TP_short_r1_780627
N1_tupi_none_long_r1_780636
N4_poti_PP_long_r1_780638
N2_poti_PP_long_r1_780630
N4_poti_PP_short_r1_780634
N4_poti_TP_short_r1_780631
N1_tupi_none_short_r1_778989
N2_poti_TP_long_r1_780629
N1_tupi_none_short_r1_780637
N4_poti_TP_long_r1_780632


### Create table

In [7]:
import os
import json
import pandas as pd
import re

# ---------- CONFIG ----------
SINGLE_NAME = 'N1_tupi_none_short_r1_778989'

SINGLE_PATH = os.path.join(BASE_DIR, SINGLE_NAME)
MULTI_PATH = BASE_DIR


# ---------- HELPERS ----------
def safe_get(d, *keys):
    """Safely extract nested dictionary values."""
    for k in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
    return d


def find_data_subfolder(base_path):
    """Find subfolder containing profile_export_aiperf.json."""
    for root, _, files in os.walk(base_path):
        if 'profile_export_aiperf.json' in files:
            return root
    return None


def load_json(path):
    try:
        with open(path, 'r') as f:
            return json.load(f)
    except Exception:
        return None


def load_jsonl_normalized(path):
    try:
        df = pd.read_json(path, lines=True)
        if df.empty:
            return pd.DataFrame()

        dfs = []
        if 'metadata' in df.columns:
            dfs.append(pd.json_normalize(df['metadata']))
        if 'metrics' in df.columns:
            dfs.append(pd.json_normalize(df['metrics']))

        return pd.concat(dfs, axis=1, join='inner') if dfs else pd.DataFrame()
    except Exception:
        return pd.DataFrame()


def extract_machine_name(log_text):
    if not log_text:
        return "Unknown"

    match = re.search(r'Node\(s\)\s*:\s*([^\n]+)', log_text)
    if match:
        return match.group(1).strip()

    match = re.search(r'SubmitHost\s*:\s*([^\n]+)', log_text)
    if match:
        return match.group(1).strip()

    return "Unknown"


# ---------- CORE LOADER ----------
def load_experiment(base_path):
    name = os.path.basename(base_path)

    data_path = find_data_subfolder(base_path)
    if not data_path:
        print(f"Skipping {name}: no profile_export_aiperf.json found")
        return None

    print(f"Processing: {name}")

    # Load core files
    profile = load_json(os.path.join(data_path, 'profile_export_aiperf.json'))
    telemetry = pd.read_csv(os.path.join(data_path, 'telemetry.csv')) \
        if os.path.exists(os.path.join(data_path, 'telemetry.csv')) else None

    jsonl_df = load_jsonl_normalized(os.path.join(data_path, 'profile_export.jsonl'))

    server_metrics = load_json(os.path.join(data_path, 'server_metrics_export.json'))
    inputs = load_json(os.path.join(data_path, 'inputs.json'))

    # Logs
    job_id = name.split('_')[-1]

    err_path_1 = os.path.join(base_path, f"{job_id}.err")
    err_path_2 = os.path.join(base_path, f"{name}.err")

    out_path_1 = os.path.join(base_path, f"{job_id}.out")
    out_path_2 = os.path.join(base_path, f"{name}.out")

    err_log = None
    out_log = None

    for p in [err_path_1, err_path_2]:
        if os.path.exists(p):
            with open(p, 'r', encoding='utf-8', errors='ignore') as f:
                err_log = f.read()
            break

    for p in [out_path_1, out_path_2]:
        if os.path.exists(p):
            with open(p, 'r', encoding='utf-8', errors='ignore') as f:
                out_log = f.read()
            break

    machine = extract_machine_name(out_log)

    return {
        'experiment_name': name,
        'profile': profile,
        'telemetry': telemetry,
        'jsonl': jsonl_df,
        'server_metrics': server_metrics,
        'inputs': inputs,
        'err': err_log,
        'out': out_log,
        'machine': machine
    }


# ---------- METRIC EXTRACTION ----------
def extract_metrics(exp):
    profile = exp['profile']
    df_jsonl = exp['jsonl']
    telemetry = exp['telemetry']

    if not profile:
        return None

    def m(name, field):
        return safe_get(profile, name, field)

    metrics = {
        'experiment_name': exp['experiment_name'],
        'machine_name': exp['machine'],

        # -------- Core metrics --------
        'request_throughput_avg': m('request_throughput', 'avg'),

        'request_latency_avg_ms': m('request_latency', 'avg'),
        'request_latency_std_ms': m('request_latency', 'std'),

        'time_to_first_token_avg_ms': m('time_to_first_token', 'avg'),
        'time_to_first_token_std_ms': m('time_to_first_token', 'std'),

        'inter_token_latency_avg_ms': m('inter_token_latency', 'avg'),
        'inter_token_latency_std_ms': m('inter_token_latency', 'std'),

        'time_to_second_token_avg_ms': m('time_to_second_token', 'avg'),
        'time_to_second_token_std_ms': m('time_to_second_token', 'std'),

        'output_token_throughput_avg': m('output_token_throughput_per_user', 'avg'),
        'output_token_throughput_std': m('output_token_throughput_per_user', 'std'),
    }

    # -------- Percentiles (expanded) --------
    if df_jsonl is not None and not df_jsonl.empty:

        def add_percentiles(col_name, prefix):
            if col_name in df_jsonl.columns:
                metrics[f'{prefix}_p50'] = df_jsonl[col_name].quantile(0.50)
                metrics[f'{prefix}_p90'] = df_jsonl[col_name].quantile(0.90)
                metrics[f'{prefix}_p99'] = df_jsonl[col_name].quantile(0.99)

        # Request-level
        add_percentiles('request_latency.value', 'request_latency_ms')

        # TTFT (time to first token)
        add_percentiles('time_to_first_token.value', 'ttft_ms')

        # Time to second token
        add_percentiles('time_to_second_token.value', 'ttst_ms')

        # Inter-token latency (token generation speed)
        add_percentiles('inter_token_latency.value', 'inter_token_latency_ms')

        # Optional: time per output token (if present)
        add_percentiles('time_per_output_token.value', 'time_per_output_token_ms')

        # Optional: total output token generation time (if exists)
        add_percentiles('output_token_time.value', 'output_token_time_ms')

    # -------- Input / Output lengths --------
    if df_jsonl is not None and 'input_sequence_length.value' in df_jsonl:
        metrics['input_sequence_length_avg'] = df_jsonl['input_sequence_length.value'].mean()
        metrics['input_sequence_length_std'] = df_jsonl['input_sequence_length.value'].std()

    if df_jsonl is not None and 'output_sequence_length.value' in df_jsonl:
        metrics['output_sequence_length_avg'] = df_jsonl['output_sequence_length.value'].mean()
        metrics['output_sequence_length_std'] = df_jsonl['output_sequence_length.value'].std()
    else:
        metrics['output_sequence_length_avg'] = m('output_sequence_length', 'avg')
        metrics['output_sequence_length_std'] = m('output_sequence_length', 'std')

    # -------- Concurrency (Little's Law) --------
    if metrics['request_throughput_avg'] and metrics['request_latency_avg_ms']:
        metrics['avg_concurrency'] = (
            metrics['request_throughput_avg'] *
            metrics['request_latency_avg_ms'] / 1000.0
        )
    else:
        metrics['avg_concurrency'] = None

    return metrics

# ---------- PIPELINE ----------
results = []

# Multi experiments (Iterando sobre o BASE_DIR)
if os.path.exists(MULTI_PATH):
    for entry in os.scandir(MULTI_PATH):
        # Filtro: processa se for diretório E não for a pasta que já processamos individualmente
        if entry.is_dir() and entry.name != SINGLE_NAME:
            exp = load_experiment(entry.path)
            if exp:
                metrics = extract_metrics(exp)
                if metrics:
                    results.append(metrics)

# Single experiment
if os.path.exists(SINGLE_PATH):
    exp = load_experiment(SINGLE_PATH)
    if exp:
        metrics = extract_metrics(exp)
        if metrics:
            results.append(metrics)
# ---------- FINAL DATAFRAME ----------
df_multi_experiment_summary = pd.DataFrame(results)

if not df_multi_experiment_summary.empty:
    print("\nSummary:")
    display(df_multi_experiment_summary)
else:
    print("No valid data found.")

Processing: N2_poti_PP_short_r1_780628
Processing: N2_poti_TP_short_r1_780627
Processing: N1_tupi_none_long_r1_780636
Processing: N4_poti_PP_long_r1_780638
Processing: N2_poti_PP_long_r1_780630
Processing: N4_poti_PP_short_r1_780634
Processing: N4_poti_TP_short_r1_780631
Processing: N2_poti_TP_long_r1_780629
Processing: N1_tupi_none_short_r1_780637
Processing: N4_poti_TP_long_r1_780632
Skipping N1_tupi_none_short_r1_778989: no profile_export_aiperf.json found

Summary:


,experiment_name,machine_name,request_throughput_avg,request_latency_avg_ms,request_latency_std_ms,time_to_first_token_avg_ms,time_to_first_token_std_ms,inter_token_latency_avg_ms,inter_token_latency_std_ms,time_to_second_token_avg_ms,...,ttst_ms_p90,ttst_ms_p99,inter_token_latency_ms_p50,inter_token_latency_ms_p90,inter_token_latency_ms_p99,input_sequence_length_avg,input_sequence_length_std,output_sequence_length_avg,output_sequence_length_std,avg_concurrency
0,N2_poti_PP_short_r1_780628,poti[3-4],0.223860,4465.753991,9.070236,78.533795,0.687966,34.545041,0.069724,34.539725,...,34.726176,35.701857,34.536224,34.566678,34.806739,128.0,0.0,128.000000,0.000000,0.999702
1,N2_poti_TP_short_r1_780627,poti[1-2],0.232459,4300.549347,12.317001,546.854121,10.256540,29.556655,0.053748,29.418769,...,29.741952,30.113029,29.534664,29.636528,29.670933,128.0,0.0,128.000000,0.000000,0.999701
2,N1_tupi_none_long_r1_780636,tupi3,0.109069,9164.210757,74.301313,110.079978,0.731509,17.866055,0.240265,17.490487,...,17.815167,17.911549,17.832636,18.137159,18.443245,1024.0,0.0,507.833333,5.233073,0.999532
3,N4_poti_PP_long_r1_780638,poti[1-4],0.049312,20273.632652,216.283287,664.567268,1.806057,38.864078,0.825291,38.109437,...,38.757086,39.085295,38.862994,39.288744,41.620206,1024.0,0.0,505.700000,8.107680,0.999729
4,N2_poti_PP_long_r1_780630,poti[3-4],0.054904,18209.715985,28.946363,396.402202,11.238016,35.178628,0.279761,34.690892,...,34.986566,35.212964,35.232774,35.479572,35.837298,1024.0,0.0,507.400000,4.214916,0.999786
5,N4_poti_PP_short_r1_780634,poti[1-4],0.209807,4764.476049,17.240032,119.998221,3.740865,36.580350,0.138056,36.499705,...,36.814227,36.960125,36.546052,36.801496,37.019746,128.0,0.0,127.966667,0.182574,0.999618
6,N4_poti_TP_short_r1_780631,poti[1-4],0.178974,5585.877742,19.985944,790.392425,2.311920,37.769710,0.167327,37.790652,...,38.110364,38.573136,37.708198,38.038721,38.222046,128.0,0.0,127.966667,0.182574,0.999724
7,N2_poti_TP_long_r1_780629,poti[1-2],0.053102,18828.286171,90.579421,3863.712939,0.859495,29.518628,0.346366,29.367977,...,29.680906,30.905465,29.425559,30.068943,30.272499,1024.0,0.0,508.000000,4.683058,0.999826
8,N1_tupi_none_short_r1_780637,tupi3,0.438948,2276.232712,13.801608,29.588125,0.756325,17.690115,0.108678,17.425317,...,17.684019,17.804670,17.737763,17.761234,17.782169,128.0,0.0,128.000000,0.000000,0.999149
9,N4_poti_TP_long_r1_780632,poti[1-4],0.040262,24833.922965,75.676834,5648.875454,7.320081,38.097763,0.698348,37.616981,...,37.752367,41.836440,37.989406,38.454965,40.654969,1024.0,0.0,504.733333,9.043700,0.999862


### Create Summarised CSV

In [8]:
df_multi_experiment_summary.to_csv("summary.csv", index=False)